# GCN Example 4 — Graph Attention Network (GAT) on the Karate Club

Same node-classification task as `gcn_karate_club.ipynb`, but with a different layer: a
**Graph Attention Network** (Veličković et al., 2018).

## GCN vs GAT — the key difference

A GCN mixes each node with its neighbours using **fixed** weights baked into the
normalized adjacency $\hat{A}$ (every neighbour's contribution is decided purely by node
degree). A GAT instead **learns how much to weight each neighbour** via an *attention*
mechanism — some friends matter more than others.

For a node $i$ with neighbours $j$:

1. Transform features: $h_j' = W h_j$.
2. Score each neighbour: $e_{ij} = \text{LeakyReLU}\big(a^\top [h_i' \,\|\, h_j']\big)$.
3. Normalize the scores over $i$'s neighbours with softmax: $\alpha_{ij} = \text{softmax}_j(e_{ij})$.
4. Aggregate: $h_i^{new} = \sigma\big(\sum_j \alpha_{ij} h_j'\big)$.

The learned $\alpha_{ij}$ are the attention weights. Because the Karate Club is small, we
compute the whole attention matrix densely and **mask** out non-edges before the softmax.

> Self-contained, runs instantly on CPU. Tested with `tensorflow==2.1`.

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

np.random.seed(0)
tf.random.set_seed(0)
print('TensorFlow', tf.__version__)

## 1. Graph, features, labels

Same setup as the GCN example: 78 edges, identity features, and only nodes 0 and 33
labelled. We build a binary adjacency **with self-loops** (so a node can also attend to
itself) and turn it into a `mask` used to block attention between non-neighbours.

In [ ]:
edges = [
    (0,1),(0,2),(0,3),(0,4),(0,5),(0,6),(0,7),(0,8),(0,10),(0,11),(0,12),
    (0,13),(0,17),(0,19),(0,21),(0,31),(1,2),(1,3),(1,7),(1,13),(1,17),(1,19),
    (1,21),(1,30),(2,3),(2,7),(2,8),(2,9),(2,13),(2,27),(2,28),(2,32),(3,7),
    (3,12),(3,13),(4,6),(4,10),(5,6),(5,10),(5,16),(6,16),(8,30),(8,32),(8,33),
    (9,33),(13,33),(14,32),(14,33),(15,32),(15,33),(18,32),(18,33),(19,33),
    (20,32),(20,33),(22,32),(22,33),(23,25),(23,27),(23,29),(23,32),(23,33),
    (24,25),(24,27),(24,31),(25,31),(26,29),(26,33),(27,33),(28,31),(28,33),
    (29,32),(29,33),(30,32),(30,33),(31,32),(31,33),(32,33),
]
N = 34
A = np.zeros((N, N), dtype='float32')
for i, j in edges:
    A[i, j] = A[j, i] = 1.0
A_self = A + np.eye(N, dtype='float32')          # allow self-attention
mask = tf.constant(A_self)                        # 1 where attention is allowed

X = tf.eye(N)
y_true = np.array([0,0,0,0,0,0,0,0,1,1,0,0,0,0,1,1,0,0,1,0,1,0,1,1,
                   1,1,1,1,1,1,1,1,1,1])
labeled_nodes = [0, 33]
y_labeled = tf.constant([y_true[0], y_true[33]])
print('Edges:', len(edges), '| labelled:', labeled_nodes)

## 2. A Graph Attention layer

We implement the dense, single-head version. The clever bit is computing all pairwise
scores cheaply: instead of forming every `[h_i' || h_j']` concatenation, we split the
attention vector `a` into `a_src` and `a_dst` and use the identity
`a^T[h_i'||h_j'] = a_src·h_i' + a_dst·h_j'`. That gives two `(N,1)` columns that
broadcast into the full `(N,N)` score matrix.

In [ ]:
class GATLayer(tf.keras.layers.Layer):
    def __init__(self, units, mask, activation=None, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.mask = mask
        self.activation = tf.keras.activations.get(activation)

    def build(self, input_shape):
        f_in = input_shape[-1]
        self.W = self.add_weight('W', (f_in, self.units), initializer='glorot_uniform')
        self.a_src = self.add_weight('a_src', (self.units, 1), initializer='glorot_uniform')
        self.a_dst = self.add_weight('a_dst', (self.units, 1), initializer='glorot_uniform')

    def call(self, x):
        h = tf.matmul(x, self.W)                       # (N, units)
        f_src = tf.matmul(h, self.a_src)               # (N, 1)
        f_dst = tf.matmul(h, self.a_dst)               # (N, 1)
        scores = tf.nn.leaky_relu(f_src + tf.transpose(f_dst), alpha=0.2)  # (N, N)
        # mask non-neighbours with a large negative value so softmax sends them to ~0
        scores = tf.where(self.mask > 0, scores, tf.fill(tf.shape(scores), -1e9))
        alpha = tf.nn.softmax(scores, axis=1)          # attention weights per row
        out = tf.matmul(alpha, h)                      # weighted neighbour aggregation
        self.alpha = alpha                              # stash for inspection
        return self.activation(out) if self.activation else out

## 3. Build and train the 2-layer GAT

Layer 1 (`34→8`, ELU) then layer 2 (`8→2`, logits). Training is identical to the GCN
example: full graph in, loss on the 2 labelled nodes only.

In [ ]:
class GAT(tf.keras.Model):
    def __init__(self, mask, hidden=8, classes=2):
        super().__init__()
        self.l1 = GATLayer(hidden, mask, activation='elu')
        self.l2 = GATLayer(classes, mask, activation=None)

    def call(self, x):
        return self.l2(self.l1(x))

model = GAT(mask)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam(0.01)

EPOCHS = 300
for epoch in range(1, EPOCHS + 1):
    with tf.GradientTape() as tape:
        logits = model(X)
        loss = loss_fn(y_labeled, tf.gather(logits, labeled_nodes))
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    if epoch % 30 == 0 or epoch == 1:
        acc = (tf.argmax(model(X), 1).numpy() == y_true).mean()
        print(f'Epoch {epoch:3d}/{EPOCHS}  loss={float(loss):.4f}  full-graph acc={acc:.3f}')

## 4. Results

In [ ]:
preds = tf.argmax(model(X), 1).numpy()
print(f'Final accuracy on all {N} nodes: {(preds == y_true).mean():.1%}')
print('Predicted:', preds)
print('True:     ', y_true)

## 5. Visualize the learned attention

Unlike a GCN, a GAT *tells us how much each neighbour mattered*. We plot the first
layer's attention matrix — bright cells are neighbour pairs the model weighted heavily.
Only edges (and self-loops) are non-zero.

In [ ]:
_ = model(X)
alpha = model.l1.alpha.numpy()
plt.figure(figsize=(6, 5))
plt.imshow(alpha, cmap='viridis')
plt.colorbar(label='attention weight')
plt.xlabel('neighbour j'); plt.ylabel('node i')
plt.title('Layer-1 attention weights (row-normalized)')
plt.show()

### Things to try

- Use **multiple attention heads** (compute several `alpha`s and concatenate their
  outputs) — this is what the original GAT does and it improves stability.
- Add dropout on the attention weights `alpha`.
- Compare final accuracy and the embedding plot against `gcn_karate_club.ipynb`.
- Inspect which neighbours node 0 vs node 33 attend to most.